In [ ]:
import pathlib
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

import torch
import trimesh
from metrics import chamfer_distance

import utils, dataset, visualization, diffusion_model

%load_ext autoreload
%autoreload 2

device = torch.device('cuda:0')

/home/nikola/miniconda3/envs/adlr/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [ ]:
#Load a model:

experiment_name = ""
config, model_config, decoder = utils.reload_model(None, None, experiment_name, 'checkpoint', device)

split = 1
match split:
    case 0:
        set_GRASP_PC = dataset.Dataset_grasp_and_pc('train')
    case 1:
        set_GRASP_PC = dataset.Dataset_grasp_and_pc('val')
    case 2:
        set_GRASP_PC = dataset.Dataset_grasp_and_pc('test')

print(len(set_GRASP_PC))

Loading val data: 100%|██████████| 232/232 [00:00<00:00, 242.03it/s]

696
696


In [ ]:
number_of_points = 2048
DDIM_steps = 50
number_of_DDIM_iterations = 1
visualize = 1

with torch.no_grad():
    
    avg_chamfer = 0.
    for object_index in range(min(len(set_GRASP_PC), 30)):
        # if object_index % 3 != 0:
        #     continue
        for i in range(number_of_DDIM_iterations):
            
            pc = set_GRASP_PC[object_index]['point_cloud'].to(device).unsqueeze(0)
            grasp = set_GRASP_PC[object_index]['grasp'].to(device).unsqueeze(0)
            

            genarated_pc = diffusion_model.sample_ddim(
                decoder, grasp, n_points=number_of_points, 
                steps=DDIM_steps, timesteps=config['timesteps']
            )
            
            # Handle Chamfer output dynamically and extract the float (.item())
            dist = chamfer_distance(pc, genarated_pc).item()
            avg_chamfer += dist
            print(f"pc genarated_pc, Object {object_index}: {dist:.4f}")
            
            if visualize:
                visualization.visualize_comparison(pc, genarated_pc, window_name="DDIM Target PC (Red) vs Generated (Blue)")
            
    print(f"Average Chamfer distance = {avg_chamfer / (len(set_GRASP_PC) * number_of_DDIM_iterations):.5f}")

pc code_pc, Object 0: 1.3304
mean_pc code_pc, Object 0: 1.0781
pc code_pc, Object 1: 0.4014
mean_pc code_pc, Object 1: 0.2955
pc code_pc, Object 2: 1.1558
mean_pc code_pc, Object 2: 1.0742
pc code_pc, Object 3: 0.1395
mean_pc code_pc, Object 3: 0.1097
pc code_pc, Object 4: 0.3794
mean_pc code_pc, Object 4: 0.3877
pc code_pc, Object 5: 83.0466
mean_pc code_pc, Object 5: 83.2434
pc code_pc, Object 6: 1.4435
mean_pc code_pc, Object 6: 1.1583


KeyboardInterrupt: 

In [3]:
import json
import pathlib
from pathlib import Path
import pybullet as p
import numpy as np
import torch

# Import your custom repository modules
import utils
import latent_decoder
import decoder  # Assumes your shape decoder.py has sample_ddim implemented

# ==========================================
# 1. CONFIGURATION (Modify as needed)
# ==========================================
BASE_DATA_DIR = Path("/home/nikola/Projects/tum-adlr-ss26-07/diffusion_autoencoder/data")
URDF_PATH = "/home/nikola/Projects/tum-adlr-ss26-07/data/studentGrasping/urdfs/dlr2.urdf"
STATS_PATH = "/home/nikola/Projects/tum-adlr-ss26-07/diffusion_autoencoder/data/normalization_stats.json"
NORM_TYPE = "global_var"  # Choices: "none", "max_norm", "global_var", "coord_var"

# Select which split folder you want to visualize: "train", "val", or "test"
DATA_SPLIT = "val"

# Model Experiment Names
VAE_EXPERIMENT_NAME = "Jun23_11-47_7000epochs_128latent_enc128_dec512_globalnorm"
LATENT_EXPERIMENT_NAME = "Jun27_15-35_epochs=2000_latent_dim=128_hidden_dim=512_embedding_dim=240"

# Generation / Inference Parameters
NUMBER_OF_POINTS = 2048
VAE_DDIM_STEPS = 50
LATENT_DDIM_STEPS = 50

# Device settings
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# ==========================================
# 2. GENERATIVE & GROUND-TRUTH VISUALIZATION
# ==========================================
def visualize_single_file_generated(npz_path, hand_id, latent_dec, vae_dec, latent_config, latent_model_config, vae_config, stats_path, norm_type):
    """
    Visualizes all grasps within an .npz file by drawing both the original 
    ground-truth point cloud (Red) and the neural-generated point cloud (Blue).
    """
    print(f"\n" + "="*60)
    print(f"Loading data from: {npz_path.name} (Comparing GT vs Generated)")
    print("="*60)
    
    try:
        data = np.load(npz_path)
        point_clouds_gt = data["point_clouds"]  # Original ground-truth cloud from dataset
        joint_angles = data["joint_angles"]
        scores = data["scores"]
    except Exception as e:
        print(f"Error loading {npz_path.name}: {e}. Skipping file.")
        return True

    num_grasps = joint_angles.shape[0]
    
    # Define colors for distinction
    gt_colors = [[0.8, 0.2, 0.2] for _ in range(NUMBER_OF_POINTS)]  # Red color for original GT PC
    gen_colors = [[0.2, 0.4, 0.8] for _ in range(NUMBER_OF_POINTS)]  # Blue color for generated PC
    
    current_gt_cloud_id = None
    current_gen_cloud_id = None

    # Load stats once for this file if normalization is active
    mean, scale = None, None
    if norm_type != "none" and stats_path and Path(stats_path).exists():
        with open(stats_path, "r") as f:
            stats = json.load(f)
        mean = np.array(stats["global_mean"])
        
        if norm_type == "max_norm":
            scale = stats["max_norm"]
        elif norm_type == "global_var":
            scale = np.sqrt(stats["global_variance"])
        elif norm_type == "coord_var":
            scale = np.sqrt(np.array(stats["coord_variance"]))

    # Iterate through grasps for this specific object
    for i in range(num_grasps):
        # Clear previous elements from the buffer
        if current_gt_cloud_id is not None:
            p.removeUserDebugItem(current_gt_cloud_id)
        if current_gen_cloud_id is not None:
            p.removeUserDebugItem(current_gen_cloud_id)

        grasp_joints = joint_angles[i]
        score = scores[i]
        orig_pc = point_clouds_gt[i]

        # A. Set finger joint angles in PyBullet environment
        for k, j in enumerate([1, 2, 3, 7, 8, 9, 13, 14, 15, 19, 20, 21]):
            p.resetJointState(
                hand_id, jointIndex=j, targetValue=grasp_joints[k], targetVelocity=0
            )
            if j in [3, 9, 15, 21]:
                p.resetJointState(
                    hand_id, jointIndex=j + 1, targetValue=grasp_joints[k], targetVelocity=0
                )

        # B. Generation Pipeline (Grasp -> Latent DDIM -> VAE DDIM -> 3D Point Cloud)
        with torch.no_grad():
            grasp_tensor = torch.tensor(grasp_joints, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            
            # Step 1: Sample intermediate latent code
            generated_code = latent_decoder.sample_ddim(
                latent_dec, 
                grasp_tensor, 
                latent_model_config['latent_dim'], 
                LATENT_DDIM_STEPS, 
                latent_config['timesteps']
            )

            # Step 2: Decode latent code into normalized 3D Point Cloud 
            generated_pc_tensor = decoder.sample_ddim(
                vae_dec, 
                generated_code, 
                n_points=NUMBER_OF_POINTS, 
                steps=VAE_DDIM_STEPS, 
                timesteps=vae_config['timesteps']
            )
            
            generated_pc = generated_pc_tensor.squeeze(0).cpu().numpy()

        # C. Reverse Global Normalization on BOTH point clouds back to original world frame scale
        if mean is not None and scale is not None:
            orig_pc = (orig_pc * scale) + mean
            generated_pc = (generated_pc * scale) + mean

        # D. Render both sets simultaneously inside PyBullet
        current_gt_cloud_id = p.addUserDebugPoints(
            pointPositions=orig_pc.tolist(),
            pointColorsRGB=gt_colors,
            pointSize=4.0 
        )
        current_gen_cloud_id = p.addUserDebugPoints(
            pointPositions=generated_pc.tolist(),
            pointColorsRGB=gen_colors,
            pointSize=4.0 
        )

        print(f" -> Grasp [{i+1}/{num_grasps}] | Score: {score:.4f} | Visualizing: Original (Red) vs Generated (Blue)")
        
        # Interactive prompt options inside Jupyter
        user_input = input("Press Enter for next grasp, 's' to skip this object, 'q' to quit entirely: ").strip().lower()
        
        if user_input in ['s', 'skip']:
            print(f"Skipping remaining grasps for object: {npz_path.name}")
            if current_gt_cloud_id is not None:
                p.removeUserDebugItem(current_gt_cloud_id)
            if current_gen_cloud_id is not None:
                p.removeUserDebugItem(current_gen_cloud_id)
            return True 
            
        elif user_input in ['q', 'quit']:
            print("Terminating visualization pipeline.")
            return False 
            
    # Clean up upon ending file loop naturally
    if current_gt_cloud_id is not None:
        p.removeUserDebugItem(current_gt_cloud_id)
    if current_gen_cloud_id is not None:
        p.removeUserDebugItem(current_gen_cloud_id)
    return True

# ==========================================
# 3. RUNTIME PIPELINE EXECUTION
# ==========================================
target_folder = BASE_DATA_DIR / DATA_SPLIT

if not target_folder.exists():
    print(f"Error: The target directory does not exist: {target_folder}")
else:
    npz_files = sorted(list(target_folder.glob("*.npz")))
    
    if not npz_files:
        print(f"No .npz files found in {target_folder}")
    else:
        print(f"Found {len(npz_files)} files to generate & visualize in folder: '{DATA_SPLIT}'")

        # Reload Trained Diffusion Weights onto Device
        print("Reloading network models onto GPU/CPU...")
        vae_config, vae_model_config, _, vae_decoder = utils.reload_model(
            None, None, VAE_EXPERIMENT_NAME, 'best_train', DEVICE
        )
        latent_config, latent_model_config, latent_decoder_model = utils.reload_model_latent(
            None, None, LATENT_EXPERIMENT_NAME, 'best_integral', DEVICE
        )

        # Initialize PyBullet Environment
        if p.isConnected():
            p.disconnect()
        p.connect(p.GUI)
        
        hand_id = p.loadURDF(
            str(URDF_PATH),
            basePosition=[0, 0, 0],
            baseOrientation=[0, 0, 0, 1],
            useFixedBase=True,
            flags=p.URDF_MAINTAIN_LINK_ORDER,
        )

        # Pipeline Loop over every target file
        try:
            for file_index, npz_path in enumerate(npz_files):
                print(f"\n[File {file_index + 1}/{len(npz_files)}]")
                
                continue_pipeline = visualize_single_file_generated(
                    npz_path, hand_id, latent_decoder_model, vae_decoder, 
                    latent_config, latent_model_config, vae_config,
                    STATS_PATH, NORM_TYPE
                )
                
                if not continue_pipeline:
                    break
        finally:
            print("\nShutting down PyBullet visualization session.")
            p.disconnect()

pybullet build time: Jan 29 2025 23:17:20


Found 232 files to generate & visualize in folder: 'val'
Reloading network models onto GPU/CPU...
startThreads creating 1 threads.
starting thread 0

[File 1/232]

Loading data from: 1071fa4cddb2da2fc8724d5673a063a6_4.npz (Comparing GT vs Generated)
started thread 0 
argc=2
argv[0] = --unused
argv[1] = --start_demo_name=Physics Server
ExampleBrowserThreadFunc started
X11 functions dynamically loaded using dlopen/dlsym OK!
X11 functions dynamically loaded using dlopen/dlsym OK!
Creating context
Created GL 3.3 context
Direct GLX rendering context obtained
Making context current
GL_VENDOR=Intel
GL_RENDERER=Mesa Intel(R) UHD Graphics 620 (KBL GT2)
GL_VERSION=4.6 (Core Profile) Mesa 25.2.8-0ubuntu0.24.04.2
GL_SHADING_LANGUAGE_VERSION=4.60
pthread_getconcurrency()=0
Version = 4.6 (Core Profile) Mesa 25.2.8-0ubuntu0.24.04.2
Vendor = Intel
Renderer = Mesa Intel(R) UHD Graphics 620 (KBL GT2)
b3Printf: Selected demo: Physics Server
startThreads creating 1 threads.
starting thread 0
started threa

In [ ]:
import json
import pathlib
from pathlib import Path
import pybullet as p
import numpy as np
import torch

# Import your custom repository modules
import utils
import diffusion_model

# ==========================================
# 1. CONFIGURATION (Modify as needed)
# ==========================================
BASE_DATA_DIR = Path("/home/nikola/Projects/tum-adlr-ss26-07/diffusion_autoencoder/data")
URDF_PATH = "/home/nikola/Projects/tum-adlr-ss26-07/data/studentGrasping/urdfs/dlr2.urdf"
STATS_PATH = "/home/nikola/Projects/tum-adlr-ss26-07/diffusion_autoencoder/data/normalization_stats.json"
NORM_TYPE = "global_var"  # Choices: "none", "max_norm", "global_var", "coord_var"

# Select which split folder you want to visualize: "train", "val", or "test"
DATA_SPLIT = "val"

# Model Experiment Names
EXPERIMENT_NAME = "Jun23_11-47_7000epochs_128latent_enc128_dec512_globalnorm"

# Generation / Inference Parameters
NUMBER_OF_POINTS = 2048
DDIM_STEPS = 50
repeat_num = 4

# Device settings
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# ==========================================
# 2. GENERATIVE VISUALIZATION FUNCTION
# ==========================================
def visualize_single_file_generated(npz_path, hand_id, dec, config, stats_path, norm_type, repeat_num):
    """
    Visualizes all grasps within an .npz file by generating the point cloud 
    conditioned on the grasp, and then un-normalizes it back to world scale.
    Repeats the generation for each grasp 'repeat_num' times.
    """
    print(f"\n" + "="*60)
    print(f"Loading data from: {npz_path.name} (Generating Point Clouds)")
    print("="*60)
    
    try:
        data = np.load(npz_path)
        joint_angles = data["joint_angles"]
        scores = data["scores"]
    except Exception as e:
        print(f"Error loading {npz_path.name}: {e}. Skipping file.")
        return True

    num_grasps = joint_angles.shape[0]
    point_colors = [[0.2, 0.4, 0.8] for _ in range(NUMBER_OF_POINTS)]  # Blue color for generated PC
    current_point_cloud_id = None

    # Load stats once for this file if normalization is active
    mean, scale = None, None
    if norm_type != "none" and stats_path and Path(stats_path).exists():
        with open(stats_path, "r") as f:
            stats = json.load(f)
        mean = np.array(stats["global_mean"])
        
        if norm_type == "max_norm":
            scale = stats["max_norm"]
        elif norm_type == "global_var":
            scale = np.sqrt(stats["global_variance"])
        elif norm_type == "coord_var":
            scale = np.sqrt(np.array(stats["coord_variance"]))

    # Iterate through grasps for this specific object
    for i in range(num_grasps):
        grasp_joints = joint_angles[i]
        score = scores[i]

        # A. Set finger joint angles in PyBullet environment (Only once per grasp configuration)
        for k, j in enumerate([1, 2, 3, 7, 8, 9, 13, 14, 15, 19, 20, 21]):
            p.resetJointState(
                hand_id, jointIndex=j, targetValue=grasp_joints[k], targetVelocity=0
            )
            if j in [3, 9, 15, 21]:
                p.resetJointState(
                    hand_id, jointIndex=j + 1, targetValue=grasp_joints[k], targetVelocity=0
                )

        # Loop to repeat generation for the SAME grasp configuration
        for rep in range(repeat_num):
            if current_point_cloud_id is not None:
                p.removeUserDebugItem(current_point_cloud_id)

            # B. Generation Pipeline (Grasp -> Latent DDIM -> VAE DDIM -> 3D Point Cloud)
            with torch.no_grad():
                grasp_tensor = torch.tensor(grasp_joints, dtype=torch.float32, device=DEVICE).unsqueeze(0)
                

                # Step 2: Decode latent code into normalized 3D Point Cloud fresh
                generated_pc_tensor = diffusion_model.sample_ddim(
                    dec, 
                    grasp_tensor, 
                    n_points=NUMBER_OF_POINTS, 
                    steps=DDIM_STEPS, 
                    timesteps=config['timesteps']
                )
                
                generated_pc = generated_pc_tensor.squeeze(0).cpu().numpy()

            # C. Reverse the Global Normalization back to original frame scale
            if mean is not None and scale is not None:
                generated_pc = (generated_pc * scale) + mean

            # D. Draw generated point cloud in PyBullet
            pts = generated_pc.tolist()
            current_point_cloud_id = p.addUserDebugPoints(
                pointPositions=pts,
                pointColorsRGB=point_colors,
                pointSize=4.0 
            )

            print(f" -> Grasp [{i+1}/{num_grasps}] | Repeat [{rep+1}/{repeat_num}] | Score: {score:.4f}")
            
            # Interactive prompt options inside Jupyter
            user_input = input("Press Enter for next step, 's' to skip this object, 'q' to quit entirely: ").strip().lower()
            
            if user_input in ['s', 'skip']:
                print(f"Skipping remaining iterations/grasps for object: {npz_path.name}")
                if current_point_cloud_id is not None:
                    p.removeUserDebugItem(current_point_cloud_id)
                return True 
                
            elif user_input in ['q', 'quit']:
                print("Terminating visualization pipeline.")
                return False 
            
    if current_point_cloud_id is not None:
        p.removeUserDebugItem(current_point_cloud_id)
    return True

# ==========================================
# 3. RUNTIME PIPELINE EXECUTION
# ==========================================
target_folder = BASE_DATA_DIR / DATA_SPLIT

if not target_folder.exists():
    print(f"Error: The target directory does not exist: {target_folder}")
else:
    npz_files = sorted(list(target_folder.glob("*.npz")))
    
    if not npz_files:
        print(f"No .npz files found in {target_folder}")
    else:
        print(f"Found {len(npz_files)} files to generate & visualize in folder: '{DATA_SPLIT}'")

        # Reload Trained Diffusion Weights onto Device
        print("Reloading network models onto GPU/CPU...")
        config, model_config, dec = utils.reload_model(
            None, None, EXPERIMENT_NAME, 'best_train', DEVICE
        )

        # Initialize PyBullet Environment
        if p.isConnected():
            p.disconnect()
        p.connect(p.GUI)
        
        hand_id = p.loadURDF(
            str(URDF_PATH),
            basePosition=[0, 0, 0],
            baseOrientation=[0, 0, 0, 1],
            useFixedBase=True,
            flags=p.URDF_MAINTAIN_LINK_ORDER,
        )

        # Pipeline Loop over every target file
        try:
            for file_index, npz_path in enumerate(npz_files):
                print(f"\n[File {file_index + 1}/{len(npz_files)}]")
                
                continue_pipeline = visualize_single_file_generated(
                    npz_path, hand_id, dec, config,
                    STATS_PATH, NORM_TYPE, repeat_num
                )
                
                if not continue_pipeline:
                    break
        finally:
            print("\nShutting down PyBullet visualization session.")
            p.disconnect()

Found 232 files to generate & visualize in folder: 'val'
Reloading network models onto GPU/CPU...
startThreads creating 1 threads.
starting thread 0

[File 1/232]

Loading data from: 1071fa4cddb2da2fc8724d5673a063a6_4.npz (Generating Point Clouds)
started thread 0 
argc=2
argv[0] = --unused
argv[1] = --start_demo_name=Physics Server
ExampleBrowserThreadFunc started
X11 functions dynamically loaded using dlopen/dlsym OK!
X11 functions dynamically loaded using dlopen/dlsym OK!
Creating context
Created GL 3.3 context
Direct GLX rendering context obtained
Making context current
GL_VENDOR=Intel
GL_RENDERER=Mesa Intel(R) UHD Graphics 620 (KBL GT2)
GL_VERSION=4.6 (Core Profile) Mesa 25.2.8-0ubuntu0.24.04.2
GL_SHADING_LANGUAGE_VERSION=4.60
pthread_getconcurrency()=0
Version = 4.6 (Core Profile) Mesa 25.2.8-0ubuntu0.24.04.2
Vendor = Intel
Renderer = Mesa Intel(R) UHD Graphics 620 (KBL GT2)
b3Printf: Selected demo: Physics Server
startThreads creating 1 threads.
starting thread 0
started thread 